# CRISPR Off-Target Mutation Analysis: Read Alignment and Variant Calling
**Organism:** Mouse (*Mus musculus*, mm10 reference genome, chromosome 2)  
**Pipeline:** BWA-MEM alignment → SAMtools processing → bcftools variant calling → Python off-target analysis

---

## Overview

CRISPR-Cas9 is a powerful gene editing tool that makes targeted cuts in the genome at sites defined by a guide RNA. However, Cas9 can also cut at genomic locations that aren't the intended target — **off-target sites** — which is a major concern for therapeutic applications where unintended edits could have harmful consequences.

This notebook implements a complete computational pipeline to detect and characterize CRISPR-induced mutations from paired-end sequencing data:

1. Align paired-end CRISPR sequencing reads to the mouse reference genome using **BWA-MEM**
2. Convert and sort the alignment using **SAMtools**
3. Call variants relative to the reference using **bcftools mpileup**
4. Filter variants by quality
5. Identify **on-target** mutations at the intended CRISPR cut sites
6. Identify and characterize **off-target** mutations across the rest of chromosome 2

**Why this matters:** off-target analysis is a required step in any CRISPR therapeutic development pipeline. Computational characterization of off-target sites from sequencing data is one of the primary ways researchers assess the specificity and safety of a new guide RNA design before moving to animal models or clinical trials.

**Reference genome:** mm10 (GRCm38), chromosome 2 only (extracted to reduce compute time)

## 1. Setup — Install Tools and Download Data

We use three standard bioinformatics command-line tools, all installable via `apt` in a Colab/Linux environment:

| Tool | Purpose |
|---|---|
| **BWA** | Aligns short sequencing reads to a reference genome using the Burrows-Wheeler Aligner (mem algorithm) |
| **SAMtools** | Converts, sorts, and indexes alignment files (SAM/BAM format) |
| **bcftools** | Calls variants (SNPs and indels) from aligned reads, with quality-based filtering |

In [ ]:
# Install alignment and variant calling tools
!apt install -y bwa samtools bcftools -q

In [ ]:
# Install cyvcf2 for parsing VCF files in Python
!pip install cyvcf2 -q

In [ ]:
# Download CRISPR paired-end sequencing reads (R1 and R2 FASTQ files)
!gdown 1-96T1PZKA_FQeD_ZK5z9USaHLP3jVdRO   # CRISPR.R1.fastq
!gdown 1-BXGr3XVGtd9Tx6PCHSp4hepK41GGCTS   # CRISPR.R2.fastq

# Download the pre-indexed mm10 reference genome (chromosome 2)
!gdown 1a8CP4P5zkzIBiw1EleqJiSwDW0VZcAar   # MM10.tar.gz

In [ ]:
# Extract the reference genome archive
# The tar archive contains the FASTA file and BWA index files (.bwt, .pac, .ann, .amb, .sa)
!tar -xzvf MM10.tar.gz

## 2. Extract and Index the Reference Chromosome

We extract only chromosome 2 from the full mm10 reference, then build a BWA index for it. Indexing creates a data structure that enables BWA to rapidly locate candidate alignment positions for each read without scanning the entire chromosome sequentially.

**Note:** the index files are already included in `MM10.tar.gz`, so the `bwa index` step below can be skipped if they extracted correctly. It's included here for completeness and reproducibility on other datasets.

In [ ]:
# Extract chromosome 2 from the full mm10 FASTA file using samtools faidx
!samtools faidx MM10/Mouse.fasta chr2 > chr2.fasta

# Confirm the extraction worked
!wc -l chr2.fasta
print("chr2.fasta written successfully")

In [ ]:
# Build BWA index for chr2
# This creates .bwt, .pac, .ann, .amb, and .sa files alongside the FASTA
# Required before alignment — skip if index files from the tar archive are already present
!bwa index chr2.fasta

## 3. Align Reads to the Reference — BWA-MEM

**BWA-MEM** (Maximal Exact Match) is the standard algorithm for aligning paired-end Illumina sequencing reads (≥70 bp) to a reference genome. It works by:
1. Finding exact seeds (short, perfect matches) between the read and the reference
2. Extending those seeds into full local alignments
3. Selecting the best alignment position per read

The output is a **SAM file** (Sequence Alignment/Map) — a tab-delimited text format recording each read's alignment position, orientation, mapping quality, and any mismatches to the reference.

In [ ]:
# Align paired-end CRISPR reads to chromosome 2
# R1 = forward reads, R2 = reverse reads from the same library
!bwa mem chr2.fasta CRISPR.R1.fastq CRISPR.R2.fastq > aligned_reads_chr2.sam

# Quick check — how many aligned reads?
!samtools flagstat aligned_reads_chr2.sam

## 4. Convert, Sort, and Index the Alignment

SAM files are human-readable but large and slow to query. We convert to **BAM** (Binary Alignment/Map) for compact storage, then sort by genomic coordinate (required for downstream variant calling), and finally index the sorted BAM so downstream tools can jump directly to any genomic region without scanning the whole file.

In [ ]:
# SAM → BAM (binary compression reduces file size ~5x)
!samtools view -Sb aligned_reads_chr2.sam > aligned_reads_chr2.bam

# Sort by genomic coordinate
!samtools sort aligned_reads_chr2.bam -o sorted_reads_chr2.bam

# Index the sorted BAM — creates sorted_reads_chr2.bam.bai
!samtools index sorted_reads_chr2.bam

print("BAM file sorted and indexed.")

In [ ]:
# Summary statistics on the final BAM file
!samtools flagstat sorted_reads_chr2.bam

## 5. Variant Calling with bcftools

**Variant calling** identifies positions in the genome where the sequencing reads differ from the reference. We use a two-step pipeline:

1. **`bcftools mpileup`** — summarizes the read pileup at every base position, computing the likelihood of observing the reads given each possible genotype
2. **`bcftools call -mv`** — uses those likelihoods to call variant sites (SNPs and indels), using the `-m` (multiallelic) model and `-v` to output variant sites only

The output is a **VCF file** (Variant Call Format), the standard format for representing genomic variants.

In [ ]:
# Step 1: mpileup — compute per-base read pileup
# Step 2: call — identify variant positions
# Piped together to avoid writing an intermediate BCF file to disk
!bcftools mpileup -Ou -f chr2.fasta sorted_reads_chr2.bam | \
    bcftools call -mv -Oz -o variants_chr2.vcf.gz

print("Variant calling complete.")

### 5.1 Quality filtering

Not all called variants are reliable — very low coverage or low base quality can produce false positives. We apply two filters:
- **QUAL > 20** — Phred-scaled quality score; QUAL=20 means a 1% probability the call is wrong (QUAL=30 = 0.1%)
- **DP > 10** — minimum read depth of 10× at the variant position, ensuring the call is based on sufficient evidence

In [ ]:
# Filter: keep only high-confidence variant calls
!bcftools filter -i 'QUAL>20 && DP>10' variants_chr2.vcf.gz -Oz -o filtered_variants_chr2.vcf.gz

# Convert to uncompressed VCF for Python parsing
!bcftools view filtered_variants_chr2.vcf.gz -Ov > filtered_variants_chr2.vcf

# Count retained variants
!grep -v '^#' filtered_variants_chr2.vcf | wc -l

## 6. Parse VCF into a DataFrame

We use `cyvcf2` — a fast Python VCF parser — to extract the key fields from each variant record and load them into a pandas DataFrame for analysis.

In [ ]:
import cyvcf2
import pandas as pd
import matplotlib.pyplot as plt

vcf_path = 'filtered_variants_chr2.vcf'

# Parse each variant record into a row
rows = []
for variant in cyvcf2.VCF(vcf_path):
    rows.append({
        'CHROM':    variant.CHROM,
        'POS':      variant.POS,
        'ID':       variant.ID,
        'REF':      variant.REF,
        'ALT':      ','.join(variant.ALT),
        'QUAL':     variant.QUAL,
        'FILTER':   variant.FILTER,
        'GENOTYPE': variant.gt_types[0]   # 0=HOM_REF, 1=HET, 2=UNKNOWN, 3=HOM_ALT
    })

df = pd.DataFrame(rows)
print(f"Total variants after QUAL>20 & DP>10 filtering: {len(df):,}")
df.head()

In [ ]:
# Apply an additional quality filter for downstream analysis
# QUAL > 60 further reduces false positives to focus on high-confidence calls
df['QUAL'] = pd.to_numeric(df['QUAL'], errors='coerce')
filtered_df = df[df['QUAL'] > 60].copy()

print(f"High-confidence variants (QUAL > 60): {len(filtered_df):,}")
print(f"\nGenotype breakdown:")
genotype_map = {0: 'HOM_REF', 1: 'HET', 2: 'UNKNOWN', 3: 'HOM_ALT'}
print(filtered_df['GENOTYPE'].map(genotype_map).value_counts())

## 7. On-Target vs. Off-Target Mutation Analysis

The **intended CRISPR cut sites** are the genomic positions where the guide RNA was designed to direct Cas9 to cut. Variants at these exact positions are expected on-target edits. Variants anywhere else in chromosome 2 are potential **off-target effects** — unintended edits at sites the guide RNA bound to imperfectly.

In [ ]:
# Known intended CRISPR cut sites on chromosome 2
intended_sites = [
    36937210, 36996899, 85400441, 85776687,
    85918029, 86198668, 86236802, 86658391, 87049235
]

# On-target: variants at intended cut sites
on_target_df = filtered_df[filtered_df['POS'].isin(intended_sites)].copy()

# Off-target: all other variants
off_target_df = filtered_df[~filtered_df['POS'].isin(intended_sites)].copy()

print(f"On-target mutations:  {len(on_target_df)}")
print(f"Off-target mutations: {len(off_target_df)}")
print(f"\nOn-target sites found:")
print(on_target_df[['POS', 'REF', 'ALT', 'QUAL', 'GENOTYPE']])

In [ ]:
# Off-target mutation table
print("Off-target mutations (first 20):")
print(off_target_df[['POS', 'REF', 'ALT', 'QUAL', 'GENOTYPE']].head(20))

## 8. Summarize and Visualize Off-Target Mutations

In [ ]:
# Summary: mutation count and unique positions per chromosome
summary_df = off_target_df.groupby('CHROM')['POS'].agg(
    mutation_count='count',
    unique_positions='nunique'
).reset_index()
summary_df.columns = ['Chromosome', 'Mutation Count', 'Unique Positions']
print(summary_df)

In [ ]:
# Distribution of off-target mutations along chromosome 2
plt.figure(figsize=(12, 5))
plt.hist(off_target_df['POS'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
plt.xlabel('Position on Chromosome 2 (bp)')
plt.ylabel('Number of Mutations')
plt.title('Off-Target CRISPR Mutations — Distribution Along Chromosome 2')
plt.tight_layout()
plt.show()

In [ ]:
# Genotype breakdown of off-target mutations
genotype_map = {0: 'HOM_REF', 1: 'Heterozygous', 2: 'Unknown', 3: 'Homozygous ALT'}
off_target_df['Genotype_Label'] = off_target_df['GENOTYPE'].map(genotype_map)

plt.figure(figsize=(7, 4))
off_target_df['Genotype_Label'].value_counts().plot(kind='bar', color='darkorange', edgecolor='black')
plt.title('Off-Target Mutation Genotype Distribution')
plt.xlabel('Genotype')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# QUAL score distribution for off-target mutations
plt.figure(figsize=(8, 4))
plt.hist(off_target_df['QUAL'].dropna(), bins=30, color='seagreen', edgecolor='white', alpha=0.8)
plt.axvline(60, color='red', linestyle='--', label='QUAL > 60 threshold')
plt.xlabel('Quality Score (QUAL)')
plt.ylabel('Count')
plt.title('Quality Score Distribution — Off-Target Mutations')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Summary

This notebook implemented a complete CRISPR off-target detection pipeline:

**Computational pipeline (shell tools):**
1. Downloaded paired-end CRISPR sequencing reads and the mm10 chromosome 2 reference
2. Aligned reads to the reference using BWA-MEM, producing a SAM file
3. Converted to BAM, sorted by coordinate, and indexed with SAMtools
4. Called variants using bcftools mpileup → bcftools call
5. Filtered variants by quality score (QUAL > 20, depth > 10, then QUAL > 60)

**Python analysis:**
6. Parsed the filtered VCF into a pandas DataFrame using cyvcf2
7. Separated on-target mutations (at intended CRISPR cut sites) from off-target mutations
8. Summarized and visualized the genomic distribution, genotype breakdown, and quality scores of off-target variants

**Key takeaways:**
- Off-target mutations from CRISPR editing are detectable computationally from sequencing data using standard variant calling pipelines
- Quality filtering (QUAL, depth thresholds) is essential before interpreting variant calls — raw calls include many low-confidence positions
- Heterozygous off-target edits are particularly meaningful: they indicate Cas9 cut only one allele at that off-target site, which is a common pattern for lower-affinity off-target binding
- This type of analysis is a standard step in CRISPR therapeutic development, where comprehensive off-target profiling is required before advancing to preclinical or clinical studies

**Potential extensions:**
- Annotate off-target mutations with gene features (coding regions, UTRs, introns) using a genome annotation file (GTF/GFF) to assess functional impact
- Compare off-target rates across multiple guide RNA designs to identify the most specific option
- Apply more sensitive off-target detection methods (e.g., GUIDE-seq, CIRCLE-seq) that are specifically designed for unbiased off-target profiling